In [5]:
# ------------------------------------------------------------
# STEP 1: LOAD DATA
# ------------------------------------------------------------

import pandas as pd
import numpy as np

url = "gs://agntworks-data-dev/wheelsup/raw/wingx/WingX_2026.zip"

df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

Shape: (885823, 18)


,FlightDate_utc,ArrivalDate_utc,Operator,FromAirport,FromCity,FromState,ToAirport,ToCity,ToState,Hours,aircraft_tailsign,aircraft_tailsign_certification,operator_type,aircraft_icao_code,aircraft_type,aircraft_model,aircraft_segment,fuel_uplift_usg
0,2026-03-31T07:44:00.000Z,2026-03-31T09:20:00.000Z,ProAir Aviation GmbH,LSGG,Geneva,NaN,LHBP,Budapest,NaN,1.600000,DCLIK,Commercial/ CAT / Part 135,Aircraft Management,C25C,Cessna-Citation CJ4,Citation CJ4 Gen2,Light Jet,307.20
1,2026-03-31T20:45:00.000Z,2026-03-31T22:01:00.000Z,CMH Services Aviation Inc,KORL,Orlando,FL,KTYS,Knoxville,TN,1.266667,N74CH,Part 91 / Non Commercial,Private Flight Department,E550,Embraer-Legacy 500 / Praetor 600,Praetor 600,Super Midsize Jet,393.93
2,2026-03-31T14:27:52.000Z,2026-03-31T15:52:44.000Z,Deer Jet,ZSFZ,Fuzhou,35,ZGSZ,Shenzhen,44,1.414444,B8302,Part 91 / Non Commercial,Aircraft Management,GLF5,Gulfstream-GV/500/550,G550,Ultra Long Range Jet,603.97
3,2026-03-31T22:12:30.000Z,2026-04-01T00:13:38.000Z,Beacon Capital Partners LLC,MMSD,Los Cabos,BCS,KSBD,San Bernardino,CA,2.018889,N119AF,Part 91 / Non Commercial,Corporate Flight Department,GLF4,Gulfstream G300/350/400/450,GIV-SP,Heavy Jet,969.07
4,2026-03-31T13:31:00.000Z,2026-03-31T14:13:00.000Z,NetJets,KDAL,Dallas,TX,KDWH,Houston,TX,0.700000,N271QS,Part 91K / Fractional Ownership,Fractional Ownership,E545,Embraer-Legacy 450 / Praetor 500,Praetor 500,Midsize Jet,210.00


In [6]:
# 1. Check for Missing Values (Nulls)
null_counts = df.isnull().sum()

# 2. Check for Unique Values (Cardinality)
unique_counts = df.nunique()

# 3. Calculate the % of uniqueness (Ratio)
# If this ratio is very low (e.g. < 5%), it's a great candidate for 'category'
unique_ratio = (df.nunique() / len(df)) * 100

# Combine into a summary table
eda_summary = pd.DataFrame({
    'Null Values': null_counts,
    'Unique Values': unique_counts,
    'Uniqueness Ratio (%)': unique_ratio.round(4)
})

print(eda_summary)

                                 Null Values  Unique Values  \
FlightDate_utc                             0         260602   
ArrivalDate_utc                            0         263201   
Operator                                   0          14560   
FromAirport                                0           6761   
FromCity                               11402           4300   
FromState                             171350            294   
ToAirport                                  0           7181   
ToCity                                 11721           4390   
ToState                               171007            297   
Hours                                      0          21898   
aircraft_tailsign                          0          22053   
aircraft_tailsign_certification            0              3   
operator_type                              0              6   
aircraft_icao_code                         0            108   
aircraft_type                              0           

In [7]:
# 'K' is the prefix for the Continental USA. 
# Anything else (C=Canada, L=Europe, M=Mexico) is International.
non_us_prefixes = df[df['ToState'].isnull()]['ToAirport'].str[0].value_counts()
print("Top 10 Airport Prefixes for Missing States:")
print(non_us_prefixes.head(10))

Top 10 Airport Prefixes for Missing States:
ToAirport
L    68437
E    38781
M    15469
T     9446
S     8957
K     5960
O     4287
R     4188
V     4143
G     2703
Name: count, dtype: int64


In [8]:
# 1. Filter for FromAirport starting with 'K' AND FromState being Null
k_null_from = df[(df['FromAirport'].str.startswith('K')) & (df['FromState'].isna())]

# 2. Get the top 10 most frequent mystery airports
top_10_k_nulls = k_null_from['FromAirport'].value_counts().head(10)

print("Top 10 US Airports (K-prefix) with missing State/City data:")
print(top_10_k_nulls)

Top 10 US Airports (K-prefix) with missing State/City data:
FromAirport
KF45    403
KT82    346
KX60    183
K27K    179
K11R    138
KF82    116
KX04    102
KM19    102
K1R8     90
K0A9     88
Name: count, dtype: int64


In [9]:
import pandas as pd

# 1. First, handle the Nulls so they are labeled correctly in the lookup
df['FromState'] = df['FromState'].fillna('International/Unknown')
df['ToState'] = df['ToState'].fillna('International/Unknown')
df['FromCity'] = df['FromCity'].fillna('Unknown')
df['ToCity'] = df['ToCity'].fillna('Unknown')

# 2. Convert Dates (This turns strings into math-friendly objects)
df['FlightDate_utc'] = pd.to_datetime(df['FlightDate_utc'])
df['ArrivalDate_utc'] = pd.to_datetime(df['ArrivalDate_utc'])

# 3. Apply the Category Mapping (The "Lookup" Magic)
# These are the columns where we replace bulk text with short reference codes
cat_columns = [
    'Operator', 'FromAirport', 'FromCity', 'FromState', 
    'ToAirport', 'ToCity', 'ToState', 'aircraft_tailsign',
    'aircraft_tailsign_certification', 'operator_type', 
    'aircraft_icao_code', 'aircraft_type', 'aircraft_model', 'aircraft_segment'
]

for col in cat_columns:
    df[col] = df[col].astype('category')

# 4. Downcast numbers (Saves 50% space on these columns)
df['Hours'] = pd.to_numeric(df['Hours'], downcast='float')
df['fuel_uplift_usg'] = pd.to_numeric(df['fuel_uplift_usg'], downcast='float')

# 5. The Moment of Truth: Check how much memory you saved
print(df.info(memory_usage='deep'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885823 entries, 0 to 885822
Data columns (total 18 columns):
 #   Column                           Non-Null Count   Dtype              
---  ------                           --------------   -----              
 0   FlightDate_utc                   885823 non-null  datetime64[ns, UTC]
 1   ArrivalDate_utc                  885823 non-null  datetime64[ns, UTC]
 2   Operator                         885823 non-null  category           
 3   FromAirport                      885823 non-null  category           
 4   FromCity                         885823 non-null  category           
 5   FromState                        885823 non-null  category           
 6   ToAirport                        885823 non-null  category           
 7   ToCity                           885823 non-null  category           
 8   ToState                          885823 non-null  category           
 9   Hours                            885823 non-null  float32  

In [10]:
# Returns an array of unique names
print(df['Operator'].unique())

['ProAir Aviation GmbH', 'CMH Services Aviation Inc', 'Deer Jet', 'Beacon Capital Partners LLC', 'NetJets', ..., 'MEXICAN NAVY-Operator', 'Jaco Oil Company', 'Falcon 399 LLC', 'High Tec Industries Services Inc', 'Pack Force One LLC']
Length: 14560
Categories (14560, object): ['01052-Operator', '01949-Operator', '0262 Aviation Inc Trustee', '04-Operator', ..., 'flyNEAT', 'iXAir', 'mLeasing SP zoo', 'nextair LLC']


In [11]:
# 1. Get flight counts for every operator
counts = df['Operator'].value_counts()

# 2. Isolate the Top 20
top_20 = counts.head(20).to_frame(name='Flight Count')

# 3. Sum up all operators from index 20 onwards
others_sum = counts.iloc[20:].sum()

# 4. Create a row for the 'Other' category
# We subtract 20 from the total length to show exactly how many operators are grouped
other_label = f"Other Operators ({len(counts) - 20:,} total)"
other_row = pd.DataFrame({'Flight Count': [others_sum]}, index=[other_label])

# 5. Combine Top 20 + Other
market_summary = pd.concat([top_20, other_row])

# 6. Calculate Market Share Percentage
total_flights = market_summary['Flight Count'].sum()
market_summary['Market Share %'] = (market_summary['Flight Count'] / total_flights * 100).round(2)

print(market_summary)

                                Flight Count  Market Share %
NetJets                               111573           12.60
Flexjet                                45233            5.11
NetJets Europe                         13520            1.53
Executive Jet Management               11094            1.25
flyExclusive                           10457            1.18
VistaJet Ltd                            8903            1.01
Vista America                           8711            0.98
Solairus Aviation                       8233            0.93
Contour Aviation                        7660            0.86
Wheels Up Private Jets                  7572            0.85
Jet Linx Aviation                       5633            0.64
Airsprint Inc                           5601            0.63
Baker Aviation LLC                      4587            0.52
Executive AirShare                      3746            0.42
Nicholas Air                            3710            0.42
Vista Germany           